In [18]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from d2l import torch as d2l

In [19]:
# Airfoil Dataset Loader

d2l.DATA_HUB["airfoil"] = (
    d2l.DATA_URL
    + "airfoil_self_noise.dat",
    "76e5be1548fd8222e5074cf0faae75edff8cf93f",
)

def get_airfoil_data(
    batch_size: int = 10,
    num_examples: int  = 1500,
) -> tuple[
    DataLoader,
    int,
]:
    file_path = d2l.download(
        "airfoil",
        folder="../data",
    )

    # [1503, 6]
    raw_data = np.genfromtxt(
        file_path,
        dtype=np.float32,
        delimiter="\t",
    )
    
    # 각 column에 대해 Mean 0, Variance 1로 Standardize
    standardized_data = (
        raw_data
        - raw_data.mean(
            axis=0,
        )
    ) / raw_data.std(
        axis=0,
    )
    
    # NumPy에서 Tensor로 변환
    data = torch.from_numpy(
        standardized_data
    )
    
    # X: [1500, 5]
    features = data[
        :num_examples,
        :-1, # 6
    ]
    
    # Y: [1500]
    targets = data[
        :num_examples,
        -1,
    ]
    
    data_loader: DataLoader = (
        d2l.load_array(
            (
                features, # [1500, 5]
                targets,  # [1500]
            ),
            batch_size,
            is_train=True,
        )
    )
    
    # 5
    num_features = features.shape[1]
    
    return (
        data_loader,
        num_features,
    )

In [20]:
# Dataset & Minibatch Shape 체크

batch_size = 10

airfoil_loader, num_features = (
    get_airfoil_data(
        batch_size=batch_size,
    )
)

dataset = airfoil_loader.dataset

if not isinstance(
    dataset,
    TensorDataset,
):
    raise TypeError(
        "Expected a TensorDataset."
    )

# [1500, 5], [1500]
all_features, all_targets = (
    dataset.tensors
)

# [10, 5], [10,]
X_batch, y_batch = next(
    iter(airfoil_loader)
)

print(
    "Full features:",
    tuple(all_features.shape),
)

print(
    "Full targets:",
    tuple(all_targets.shape),
)

print(
    "Number of features:",
    num_features,
)

print(
    "Batch features:",
    tuple(X_batch.shape),
)

print(
    "Batch targets:",
    tuple(y_batch.shape),
)

Full features: (1500, 5)
Full targets: (1500,)
Number of features: 5
Batch features: (10, 5)
Batch targets: (10,)


In [21]:
# Standardization 검증

feature_means = all_features.mean(
    dim=0,
)

feature_standard_deviations = (
    all_features.std(
        dim=0,
        correction=0,
    )
)

print(
    "Feature means:",
    feature_means,
)

print(
    "Feature standard deviations:",
    feature_standard_deviations,
)

print(
    "Target mean:",
    all_targets.mean().item(),
)

print(
    "Target standard deviation:",
    all_targets.std(
        correction=0,
    ).item(),
)

Feature means: tensor([-0.0014, -0.0030,  0.0008,  0.0014, -0.0063])
Feature standard deviations: tensor([1.0004, 0.9988, 1.0009, 1.0005, 0.9909])
Target mean: 0.005558027420192957
Target standard deviation: 0.9932205677032471
